# AI PDF Study Assistant — Prototype

End-to-end pipeline: **PDF → text → chunks → AI-generated learning assets**.

Every step is implemented in `study_assistant/` so the same code can later be
ported into Django services. The notebook is the **inspection layer**:
run cells top-to-bottom and look at intermediate outputs.

## Sections
1. Setup  
2. Load PDF  
3. Extract Text  
4. Chunk Text  
5. Generate Summaries  
6. Generate Flashcards  
7. Generate Quizzes  
8. Extract Concepts  
9. Generate Mind Map  
10. Generate Audio Script  
11. Export Results  
12. Evaluation


## 1. Setup

Load env vars, configure paths, build the cached LLM client.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# Make the prototype package importable when running from notebooks/.
PROTOTYPE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROTOTYPE_DIR))

from dotenv import load_dotenv
load_dotenv(PROTOTYPE_DIR / '.env')

from study_assistant.cache import JSONCache
from study_assistant.llm_client import LLMClient, LLMConfig

CACHE_DIR    = PROTOTYPE_DIR / 'cache'
OUTPUTS_DIR  = PROTOTYPE_DIR / 'outputs'
SAMPLE_DIR   = PROTOTYPE_DIR / 'sample_pdfs'

for sub in ('summaries', 'flashcards', 'quizzes', 'mindmaps', 'audio_scripts', 'json', 'notes', 'html'):
    (OUTPUTS_DIR / sub).mkdir(parents=True, exist_ok=True)

cache  = JSONCache(CACHE_DIR)
config = LLMConfig.from_env()
client = LLMClient(config=config, cache=cache)

print(f'Provider : {config.provider}')
print(f'Model    : {config.model}')
print(f'Cache    : {CACHE_DIR}')
print(f'Outputs  : {OUTPUTS_DIR}')


## 2. Load PDF

Drop a textbook PDF into `prototype/sample_pdfs/` and set its name below.


In [ ]:
PDF_NAME = 'textbook.pdf'   # <-- change to your file name
PDF_PATH = SAMPLE_DIR / PDF_NAME

assert PDF_PATH.exists(), f'PDF not found: {PDF_PATH}. Drop a file into {SAMPLE_DIR}.'
print(f'Using PDF: {PDF_PATH}  ({PDF_PATH.stat().st_size / 1024:.1f} KB)')


## 3. Extract Text

PyMuPDF backend, with header/footer/page-number cleanup.


In [ ]:
from study_assistant.pdf_extractor import extract_pdf_text

pdf = extract_pdf_text(PDF_PATH, backend='pymupdf', drop_headers_footers=True)

print(f'Pages extracted: {len(pdf.pages)}')
print(f'Total characters: {sum(len(p.text) for p in pdf.pages):,}')
print()
print('--- Page 1 preview ---')
print(pdf.pages[0].text[:800])


## 4. Chunk Text

Paragraph-aware chunking targeting ~1500 words with overlap.


In [ ]:
from study_assistant.chunker import chunk_extracted_pdf

chunks = chunk_extracted_pdf(
    pdf,
    target_words=1500,
    min_words=1000,
    max_words=2000,
    overlap_words=150,
)

print(f'Total chunks: {len(chunks)}')
for c in chunks[:5]:
    print(f'  Chunk {c.chunk_id:>3}: {c.word_count:>5} words  (pages {c.start_page}-{c.end_page})')
if len(chunks) > 5:
    print(f'  ... and {len(chunks) - 5} more')


In [ ]:
# For fast iteration during development, optionally sub-sample chunks.
# Set MAX_CHUNKS to None to process the whole book.
MAX_CHUNKS = 3

work_chunks = chunks if MAX_CHUNKS is None else chunks[:MAX_CHUNKS]
print(f'Will process {len(work_chunks)} / {len(chunks)} chunks.')


## 5. Generate Summaries


In [ ]:
from tqdm.auto import tqdm
from study_assistant.summary_service import summarise_chunk

summaries = [summarise_chunk(client, c) for c in tqdm(work_chunks, desc='Summaries')]

s = summaries[0]
print(f'--- Chunk {s.chunk_id} ---')
print('Short :', s.summary_short)
print()
print('Long  :', s.summary_long[:600], '...')
print()
print('Takeaways:')
for t in s.takeaways:
    print(f'  - {t}')


## 6. Generate Flashcards


In [ ]:
from study_assistant.flashcard_service import generate_flashcards

flashcard_sets = [generate_flashcards(client, c) for c in tqdm(work_chunks, desc='Flashcards')]

total_cards = sum(len(fs.flashcards) for fs in flashcard_sets)
print(f'Total flashcards generated: {total_cards}')
print('\n--- Sample (chunk 1) ---')
for i, card in enumerate(flashcard_sets[0].flashcards[:5], 1):
    print(f'Q{i}. {card.question}')
    print(f'A.  {card.answer}\n')


## 7. Generate Quizzes


In [ ]:
from study_assistant.quiz_service import generate_quiz

quiz_sets = [generate_quiz(client, c) for c in tqdm(work_chunks, desc='Quizzes')]

total_q = sum(len(qs.questions) for qs in quiz_sets)
print(f'Total quiz questions generated: {total_q}')
print('\n--- Sample (chunk 1) ---')
for q in quiz_sets[0].questions[:3]:
    print(f'[{q.type}] {q.question}')
    for opt in q.options:
        print(f'    - {opt}')
    print(f'    Answer    : {q.correct_answer}')
    print(f'    Explanation: {q.explanation}\n')


## 8. Extract Concepts

Per-chunk extraction, then de-duplicated across the whole book.


In [ ]:
from study_assistant.concept_service import extract_concepts, merge_concepts

concept_sets = [extract_concepts(client, c) for c in tqdm(work_chunks, desc='Concepts')]
merged = merge_concepts(concept_sets)

print(f'Unique concepts   : {len(merged.concepts)}')
print(f'Unique definitions: {len(merged.definitions)}')
print('\n--- Top concepts ---')
for c in merged.concepts[:10]:
    print(f'- {c.name}: {c.description}')


## 9. Generate Mind Map

Built from the merged concept list to keep the prompt compact.


In [ ]:
from study_assistant.mindmap_service import generate_mindmap, render_mindmap

material = '\n'.join(f'- {c.name}: {c.description}' for c in merged.concepts)
mindmap  = generate_mindmap(client, material, cache_tag=f'mindmap:{PDF_NAME}')

print(json.dumps(mindmap.model_dump(), indent=2)[:800])


In [ ]:
from IPython.display import Image, display

png_path = render_mindmap(
    mindmap,
    OUTPUTS_DIR / 'mindmaps' / f'{PDF_PATH.stem}_mindmap.png',
    fmt='png',
)
print(f'Mind map saved to: {png_path}')
display(Image(filename=str(png_path)))


## 10. Generate Audio Script

10-15 minute spoken lesson based on the chunk summaries. TTS synthesis is optional.


In [ ]:
from study_assistant.audio_service import generate_audio_script

lesson_material = '\n\n'.join(
    f'### Chunk {s.chunk_id}\n{s.summary_long}' for s in summaries
)

script = generate_audio_script(client, lesson_material, cache_tag=f'audio:{PDF_NAME}')

script_path = OUTPUTS_DIR / 'audio_scripts' / f'{PDF_PATH.stem}_lesson.md'
script_path.write_text(script, encoding='utf-8')
print(f'Saved: {script_path}  ({len(script.split())} words)')
print('\n--- preview ---\n')
print(script[:800])


In [ ]:
# Optional: synthesize the script to MP3 using OpenAI TTS.
# Requires OPENAI_API_KEY and the openai package.
SYNTHESIZE_AUDIO = False

if SYNTHESIZE_AUDIO:
    from study_assistant.audio_service import script_to_speech_openai
    mp3_path = script_to_speech_openai(
        script,
        OUTPUTS_DIR / 'audio_scripts' / f'{PDF_PATH.stem}_lesson.mp3',
    )
    print(f'Saved: {mp3_path}')
else:
    print('Audio synthesis skipped. Set SYNTHESIZE_AUDIO = True to enable.')


## 11. Export Results

Write JSON, Markdown, and an HTML report into `prototype/outputs/`.


In [ ]:
from study_assistant import exporter
from study_assistant.notes_service import generate_notes

stem = PDF_PATH.stem

# --- JSON dumps -----------------------------------------------------
exporter.dump_json(summaries,      OUTPUTS_DIR / 'json' / f'{stem}_summaries.json')
exporter.dump_json(flashcard_sets, OUTPUTS_DIR / 'json' / f'{stem}_flashcards.json')
exporter.dump_json(quiz_sets,      OUTPUTS_DIR / 'json' / f'{stem}_quizzes.json')
exporter.dump_json(merged,         OUTPUTS_DIR / 'json' / f'{stem}_concepts.json')
exporter.dump_json(mindmap,        OUTPUTS_DIR / 'json' / f'{stem}_mindmap.json')

# --- Markdown views -------------------------------------------------
summaries_md  = exporter.summaries_to_markdown(summaries)
flashcards_md = exporter.flashcards_to_markdown(flashcard_sets)
quizzes_md    = exporter.quizzes_to_markdown(quiz_sets)
concepts_md   = exporter.concepts_to_markdown(merged)

exporter.write_text(summaries_md,  OUTPUTS_DIR / 'summaries'  / f'{stem}_summaries.md')
exporter.write_text(flashcards_md, OUTPUTS_DIR / 'flashcards' / f'{stem}_flashcards.md')
exporter.write_text(quizzes_md,    OUTPUTS_DIR / 'quizzes'    / f'{stem}_quizzes.md')
exporter.write_text(concepts_md,   OUTPUTS_DIR / 'summaries'  / f'{stem}_concepts.md')

# --- Study notes (uses summarised material to control prompt size) ---
notes_material = '\n\n'.join(s.summary_long for s in summaries)
notes_md = generate_notes(client, topic=stem.replace('_', ' ').title(),
                          material=notes_material, cache_tag=f'notes:{PDF_NAME}')
exporter.write_text(notes_md, OUTPUTS_DIR / 'notes' / f'{stem}_notes.md')

# --- HTML report ----------------------------------------------------
import markdown as md_renderer  # provided by jupyter deps; falls back below if missing
try:
    to_html = lambda s: md_renderer.markdown(s, extensions=['extra'])
except Exception:
    to_html = lambda s: '<pre>' + s.replace('<', '&lt;') + '</pre>'

html = exporter.html_report(
    title=f'Study pack — {stem}',
    sections={
        'Summaries':   to_html(summaries_md),
        'Flashcards':  to_html(flashcards_md),
        'Quizzes':     to_html(quizzes_md),
        'Concepts':    to_html(concepts_md),
        'Study Notes': to_html(notes_md),
    },
)
html_path = OUTPUTS_DIR / 'html' / f'{stem}_report.html'
exporter.write_text(html, html_path)
print(f'HTML report: {html_path}')


## 12. Evaluation

Aggregate statistics for this run.


In [ ]:
stats = exporter.build_statistics(
    n_pages=len(pdf.pages),
    n_chunks=len(work_chunks),
    flashcard_sets=flashcard_sets,
    quiz_sets=quiz_sets,
    merged_concepts=merged,
)

stats_path = OUTPUTS_DIR / 'json' / f'{PDF_PATH.stem}_statistics.json'
exporter.dump_json(stats, stats_path)

print(json.dumps(stats.model_dump(), indent=2))
print(f'\nStatistics saved: {stats_path}')


---

### Success checklist

- [ ] PDF text extracted cleanly (no header/footer noise)
- [ ] Summaries read as coherent paragraphs
- [ ] Flashcards are useful for revision (no trivia)
- [ ] Quiz questions are answerable from the chunk
- [ ] Mind map reflects actual topic structure
- [ ] Audio script sounds like a mini-lecture when read aloud
- [ ] All assets exported to `outputs/`
